# 1. One material

This tutorial assesses the emissions of making one material with
[`assess()`](../api/result.md#dlca.result.assess), then reads the three
kinds of result: global warming impact (GWI), temperature change (ΔT) and
global temperature change potential (GTP). It ends by running the same
emissions with the AR5 and the AR6 parameter sets.

!!! note "Illustrative numbers"
    The material is the concrete slab of the example file
    [`tests/data/inventories/concrete.json`](https://github.com/jayarehart/DLCA/tree/main/tests/data/inventories).
    Every number in that file is made up. The results below show how to use
    the package; they are not findings about concrete.

In [ ]:
import json
from pathlib import Path

# The example inventories live in the repository's tests/data/inventories/.
# Look for them from the folder this notebook runs in, upwards.
here = Path.cwd().resolve()
DATA = next(
    p / "tests" / "data" / "inventories"
    for p in [here, *here.parents]
    if (p / "tests" / "data" / "inventories").is_dir()
)


def load(name):
    """The example file `name`.json as a plain dict."""
    return json.loads((DATA / f"{name}.json").read_text(encoding="utf-8"))

## Build an emission series

`assess()` takes kilograms of each gas per year, with index 0 as year 0. Here
the A1–A3 (product stage) amounts of 250 m³ of concrete are all emitted in
year 0, followed by 100 years of zeros so the series is long enough to read
at a 100-year horizon.

In [ ]:
slab = load("concrete")["assemblies"][0]["materials"][0]
quantity = slab["quantity"]  # m3
per_m3 = slab["modules"]["A1-A3"]  # kg of each gas per m3
print(slab["name"], quantity, slab["unit"], per_m3)

years = 101  # years 0..100
emissions = {gas: [0.0] * years for gas in per_m3}
for gas, amount in per_m3.items():
    emissions[gas][0] = amount * quantity

{gas: series[0] for gas, series in emissions.items()}

## Assess it

The CH₄ here is `ch4_fossil`: methane from fossil supply chains, which also
turns into CO₂ in the air. Non-fossil methane goes under `ch4`.

In [ ]:
from dlca import assess

result = assess(emissions, parameters="AR6", horizon=100)
result.parameter_set, result.horizon

## Read the results

Every series has one value per year. Three of them:

- `result.gwi.dynamic[t]`: the warming effect (radiative forcing) added up
  from year 0 to year t, expressed as the kg of CO₂ emitted in year 0 that
  would add up to the same over the horizon [kg CO₂e].
- `result.temperature.total[t]`: the change in global mean surface
  temperature in year t [K].
- `result.gtp.dynamic[t]`: that temperature change expressed as kg CO₂e.

In [ ]:
import pandas as pd

picked = [1, 10, 20, 50, 100]
pd.DataFrame(
    {
        "GWI dynamic [kg CO2e]": result.gwi.dynamic[picked],
        "ΔT [K]": result.temperature.total[picked],
        "GTP dynamic [kg CO2e]": result.gtp.dynamic[picked],
    },
    index=pd.Index(picked, name="year"),
)

GWI keeps growing, because forcing keeps adding up. GTP follows the
temperature in each year. The temperature change of one slab is tiny in
kelvin; the CO₂e units make it comparable with other emissions.

The static result is the familiar GWP-100 sum: each gas's mass times its
GWP, with no timing. For one pulse in year 0, the dynamic GWI at year 100
should be close to it.

In [ ]:
print("static GWP-100 [kg CO2e]:", round(result.static_gwp.total[-1], 1))
print("dynamic GWI at year 100 [kg CO2e]:", round(result.gwi.dynamic[100], 1))
print("GWP-100 used, by gas:", {k: round(v, 2) for k, v in result.static_gwp.gwp.items()})

## AR5 against AR6

The same emissions with the two packaged parameter sets. The shipped AR6
set includes the carbon-cycle feedback and AR5 does not, so the numbers
differ a little.

In [ ]:
from dlca import available_parameter_sets

results = {
    name: assess(emissions, parameters=name, horizon=100) for name in available_parameter_sets()
}

pd.DataFrame(
    {
        name: {
            "GWI dynamic, year 100 [kg CO2e]": r.gwi.dynamic[100],
            "GTP dynamic, year 100 [kg CO2e]": r.gtp.dynamic[100],
            "Static GWP-100 [kg CO2e]": r.static_gwp.total[-1],
            "Carbon-cycle feedback": r.carbon_cycle_feedback,
        }
        for name, r in results.items()
    }
)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
for name, r in results.items():
    ax1.plot(r.gwi.dynamic / 1000, label=name)
    ax2.plot(r.gtp.dynamic / 1000, label=name)
ax1.set(title="GWI, dynamic", xlabel="year", ylabel="t CO2e")
ax2.set(title="GTP, dynamic", xlabel="year", ylabel="t CO2e")
ax1.legend()
fig.tight_layout()

For this slab the two sets give curves of the same shape and values within
about 0.3 % of each other at year 100. Which set to use is a modelling
choice, not something this tutorial decides.

Next: [an assembly over its life](02_assembly.ipynb).